# Mortgage Document RAG Gradio App

## Objective
Implements an end-to-end Gradio interface for uploading mortgage documents, extracting text, indexing chunks, and asking document questions.

## Approach
- Install stable Colab dependencies
- Extract text with PyMuPDF and OCR fallback
- Embed chunks with BGE and search with FAISS
- Answer questions through a Gradio UI

## Expected Result
This notebook turns the externship pipeline into a working demo application.

## Running in Google Colab
These notebooks were developed in Google Colab. For reproducibility, place required PDFs in a `data/` folder when running locally, or upload them to the Colab working directory. The helper functions below try common Colab and GitHub-style paths.

## Security Note
API keys are not stored in the notebook. Use Colab Secrets with the name `GOOGLE_API_KEY` or set the environment variable manually.

## Project Context
This notebook is part of a curated document intelligence externship portfolio project completed through Outamation. The work focuses on OCR, document parsing, retrieval, LLM-based question answering, and prototype application development for mortgage-style document analysis.

## Data Note
The notebooks were originally developed in Google Colab. Any document files used for testing should be placed in the `data/` folder or uploaded directly into the Colab runtime. The sample documents used for this educational project do not contain sensitive personal information.


In [ ]:

# -------------------------
# INSTALLS
# -------------------------
!apt-get -y update >/dev/null 2>&1
!apt-get -y install tesseract-ocr poppler-utils >/dev/null 2>&1

!pip install -q pymupdf pillow pytesseract faiss-cpu numpy
!pip install -q sentence-transformers==3.0.1
!pip install -q transformers==4.46.0
!pip install -q accelerate
!pip install -q langchain==0.3.0
!pip install -q langchain-text-splitters==0.3.0
!pip install -U gradio gradio_client


In [ ]:
# ============================================================
# FULL RAG UI FOR MORTGAGE DOCUMENTS - STABLE COLAB VERSION
# ============================================================


# -------------------------
# 1. IMPORTS
# -------------------------
import os
import re
import warnings
from dataclasses import dataclass
from typing import List, Dict, Tuple

warnings.filterwarnings("ignore")

import numpy as np
import fitz
import pytesseract
import faiss
import gradio as gr
from PIL import Image
import torch

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer
from transformers import pipeline

print("✅ All imports successful!")
print("Gradio version:", gr.__version__)

# IMPORTANT:
# If this prints an old version or type='messages' still errors,
# restart the runtime once after installs, then run the cell again.

# -------------------------
# 2. MODEL CONFIG
# -------------------------
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"
GEN_MODEL_NAME = "google/flan-t5-small"

DEFAULT_TOP_K = 3
MAX_CONTEXT_CHARS = 800
MIN_DIGITAL_TEXT_CHARS = 80

# -------------------------
# 3. LOAD MODELS
# -------------------------
print("Loading embedding model...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

print("Loading generation model...")
device = 0 if torch.cuda.is_available() else -1

generator = pipeline(
    task="text2text-generation",
    model=GEN_MODEL_NAME,
    device=device
)

print(f"✅ Models loaded! Using {'GPU' if device == 0 else 'CPU'} for generation.")

# -------------------------
# 4. DATA STRUCTURES
# -------------------------
@dataclass
class ChunkRecord:
    text: str
    source_file: str
    page_num: int
    doc_type: str
    page_label: str

# -------------------------
# 5. TEXT PROCESSING
# -------------------------
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"(?<=\w)-\n(?=\w)", "", text)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    return text.strip()

def guess_doc_type(text: str, filename: str) -> str:
    t = f"{filename}\n{text[:2000]}".lower()
    if any(k in t for k in ["loan estimate", "closing cost", "apr", "cash to close"]):
        return "loan_estimate"
    if any(k in t for k in ["mortgage", "deed of trust", "borrower", "lender"]):
        return "mortgage_form"
    if any(k in t for k in ["pay stub", "earnings", "net pay"]):
        return "pay_stub"
    return "general"

def ocr_page(page) -> str:
    pix = page.get_pixmap(matrix=fitz.Matrix(2, 2), alpha=False)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    text = pytesseract.image_to_string(img)
    return clean_text(text)

def extract_pages_from_pdf(pdf_path: str) -> List[Dict]:
    doc = fitz.open(pdf_path)
    filename = os.path.basename(pdf_path)
    pages = []

    try:
        for i in range(len(doc)):
            page = doc[i]
            digital_text = clean_text(page.get_text("text"))

            if len(digital_text) < MIN_DIGITAL_TEXT_CHARS:
                page_text = ocr_page(page)
            else:
                page_text = digital_text

            pages.append({
                "page_num": i + 1,
                "text": page_text,
                "source_file": filename,
            })
    finally:
        doc.close()

    return pages

# -------------------------
# 6. CHUNKING
# -------------------------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

def build_chunks_from_files(file_paths) -> List[ChunkRecord]:
    all_chunks = []

    for pdf_path in file_paths:
        filename = os.path.basename(pdf_path)

        try:
            pages = extract_pages_from_pdf(pdf_path)
        except Exception as e:
            print(f"⚠️ Failed to read {filename}: {e}")
            continue

        for page in pages:
            page_text = page["text"]
            if len(page_text) < 30:
                continue

            doc_type = guess_doc_type(page_text, filename)
            chunks = splitter.split_text(page_text)

            for chunk in chunks:
                chunk = clean_text(chunk)
                if len(chunk) < 50:
                    continue

                all_chunks.append(
                    ChunkRecord(
                        text=chunk,
                        source_file=filename,
                        page_num=page["page_num"],
                        doc_type=doc_type,
                        page_label=f"{filename} p.{page['page_num']}"
                    )
                )

    return all_chunks

# -------------------------
# 7. VECTOR INDEX
# -------------------------
def build_index(chunks: List[ChunkRecord]):
    if not chunks:
        return None

    texts = [c.text for c in chunks]
    embeddings = embed_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    embeddings = np.asarray(embeddings, dtype="float32")

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index

def retrieve(query: str, chunks: List[ChunkRecord], index, top_k: int = DEFAULT_TOP_K):
    if not chunks or index is None:
        return []

    q_emb = embed_model.encode([query], normalize_embeddings=True)
    q_emb = np.asarray(q_emb, dtype="float32")

    scores, indices = index.search(q_emb, min(top_k * 2, len(chunks)))

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append((float(score), chunks[idx]))
        if len(results) >= top_k:
            break

    return results

# -------------------------
# 8. ANSWER GENERATION
# -------------------------
def generate_answer(query: str, retrieved_chunks: List[Tuple[float, ChunkRecord]]) -> str:
    if not retrieved_chunks:
        return "I could not find relevant information in the uploaded documents."

    context_parts = []
    for score, chunk in retrieved_chunks[:3]:
        context_parts.append(f"[{chunk.page_label}] {chunk.text[:MAX_CONTEXT_CHARS]}")

    context = "\n\n".join(context_parts)

    prompt = f"""Answer the question based ONLY on the context below.
If the answer is not in the context, say: "I cannot find this information in the documents."

Context:
{context}

Question: {query}

Answer:"""

    try:
        result = generator(
            prompt,
            max_new_tokens=120,
            do_sample=False
        )[0]["generated_text"]
        return result.strip()
    except Exception as e:
        return f"Error generating answer: {e}"

# -------------------------
# 9. GLOBAL STATE
# -------------------------
class RAGState:
    def __init__(self):
        self.chunks = []
        self.index = None
        self.is_ready = False

rag_state = RAGState()

# -------------------------
# 10. APP FUNCTIONS
# -------------------------
def process_pdfs(files):
    if not files:
        return "Please upload PDF files first.", "No files uploaded.", []

    file_paths = files if isinstance(files, list) else [files]
    valid_paths = [f for f in file_paths if f and str(f).lower().endswith(".pdf")]

    if not valid_paths:
        return "No valid PDF files found.", "Please upload PDF files only.", []

    chunks = build_chunks_from_files(valid_paths)
    if not chunks:
        return "No text could be extracted from the PDFs.", "Extraction failed.", []

    index = build_index(chunks)

    rag_state.chunks = chunks
    rag_state.index = index
    rag_state.is_ready = True

    unique_files = sorted(set(c.source_file for c in chunks))
    doc_types = sorted(set(c.doc_type for c in chunks))

    status = (
        f"✅ Processed {len(chunks)} chunks from {len(unique_files)} file(s).\n"
        f"Detected document types: {', '.join(doc_types)}"
    )
    sources = "Files loaded:\n" + "\n".join(unique_files)

    return status, sources, []

def ask_question(query, history):
    history = history or []

    if not query or not query.strip():
        return history, "Please enter a question.", ""

    if not rag_state.is_ready:
        history = history + [{"role": "assistant", "content": "Please upload and process PDFs first."}]
        return history, "No documents processed yet.", ""

    try:
        retrieved = retrieve(query, rag_state.chunks, rag_state.index, top_k=DEFAULT_TOP_K)
        answer = generate_answer(query, retrieved)

        sources_text = "Retrieved sources:\n"
        if retrieved:
            for i, (score, chunk) in enumerate(retrieved, 1):
                sources_text += f"{i}. {chunk.page_label} | {chunk.doc_type} | relevance: {score:.3f}\n"
        else:
            sources_text += "No matching chunks found."

        history = history + [
            {"role": "user", "content": query},
            {"role": "assistant", "content": answer}
        ]

        return history, sources_text, ""

    except Exception as e:
        history = history + [
            {"role": "user", "content": query},
            {"role": "assistant", "content": f"Error: {str(e)}"}
        ]
        return history, f"Error: {str(e)}", ""

def clear_chat():
    return [], "", ""

# -------------------------
# 11. UI
# -------------------------
with gr.Blocks(title="Mortgage Document RAG", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Mortgage Document Assistant")
    gr.Markdown("Upload mortgage-related PDFs, process them, and ask questions.")

    chat_state = gr.State([])

    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(
                label="Upload PDFs",
                file_types=[".pdf"],
                file_count="multiple",
                type="filepath"
            )
            process_btn = gr.Button("Process Documents")
            status_output = gr.Textbox(label="Status", lines=4)
            sources_output = gr.Textbox(label="Retrieved Sources / Loaded Files", lines=10)

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Chat", height=450)
            question_input = gr.Textbox(
                label="Your Question",
                placeholder="e.g., What is the loan amount?",
                lines=2
            )

            with gr.Row():
                ask_btn = gr.Button("Ask")
                clear_btn = gr.Button("Clear")

    process_btn.click(
        fn=process_pdfs,
        inputs=[file_input],
        outputs=[status_output, sources_output, chat_state]
    ).then(
        fn=lambda hist: hist,
        inputs=[chat_state],
        outputs=[chatbot]
    )

    ask_btn.click(
        fn=ask_question,
        inputs=[question_input, chat_state],
        outputs=[chat_state, sources_output, question_input]
    ).then(
        fn=lambda hist: hist,
        inputs=[chat_state],
        outputs=[chatbot]
    )

    question_input.submit(
        fn=ask_question,
        inputs=[question_input, chat_state],
        outputs=[chat_state, sources_output, question_input]
    ).then(
        fn=lambda hist: hist,
        inputs=[chat_state],
        outputs=[chatbot]
    )

    clear_btn.click(
        fn=clear_chat,
        inputs=None,
        outputs=[chat_state, sources_output, question_input]
    ).then(
        fn=lambda hist: hist,
        inputs=[chat_state],
        outputs=[chatbot]
    )

print("🚀 Launching interface...")
demo.launch(debug=True, share=True)
